# 05 Extensions

This notebook contains exploratory simulations and future-work concepts that are
excluded from the manuscript's primary evidence chain. In particular, the vector
timer requires stage-level readout, the capture transition is assumed rather than
measured, and the device-native actor--critic is a repository-specific construction.
Every numerical panel is computed from its stated simulation model. The
default `reduced` profile uses fewer independent samples and a coarser integration
step than the full study, while retaining the authored plotting code and axes.
Archived full sweeps are available only through the explicit opt-in below.  Figures
always display inline; saving is disabled by default.

## Configuration

The controls match the other publication notebooks. NumPy reference algorithms stay
on CPU and independent cells are distributed across spawn-safe processes. A child
notebook launched by `REPRODUCE.ipynb` disables its inner pool to prevent nesting.

In [ ]:
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context
from IPython.display import display
import os

for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_name, "1")

import numpy as np
import matplotlib.pyplot as plt

RUN_PROFILE = os.getenv("MRL_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MRL_DEVICE", "auto")               # CPU for these NumPy reference algorithms
WORKERS = os.getenv("MRL_WORKERS", "auto")
SAVE_FIGURES = os.getenv("MRL_SAVE_FIGURES", "0") == "1"
OUTPUT_DIR = Path(os.getenv("MRL_OUTPUT_DIR", "generated_figures"))
OVERWRITE = os.getenv("MRL_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MRL_RUN_EXTERNAL_DATA", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MRL_ALLOW_DATA_DOWNLOADS", "0") == "1"
USE_ARCHIVED_RESULTS = os.getenv("MRL_USE_ARCHIVED_RESULTS", "0") == "1"

if RUN_PROFILE not in {"reduced", "publication", "smoke"}:
    raise ValueError(f"unknown RUN_PROFILE={RUN_PROFILE!r}")

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "experiments" else HERE
OWNED = ("fig_beta_sensitivity.png", "fig_betaval.png", "fig_capture.png",
         "fig_device_td.png", "fig_interval.png", "fig_long_horizon.png",
         "fig_multitimescale.png", "fig_reversal.png",
         "fig_vector_timer.png", "fig_wm_stc.png")
FIGURE_REPORT = []

if os.getenv("MRL_CHILD_PROCESS") == "1":
    RESOLVED_WORKERS = 1
elif WORKERS == "auto":
    RESOLVED_WORKERS = max(1, min(6, (os.cpu_count() or 4) - 2))
else:
    RESOLVED_WORKERS = max(1, int(WORKERS))

SETTINGS = {
    "smoke": dict(seeds=1, reversal_seeds=1, vector_seeds=1, long_seeds=1,
                  interval_trials=5, reversal_trials=40, multi_trials=20,
                  vector_trials=10, vector_reps=80, wm_trials=20, td_episodes=12,
                  beta_episodes=10, beta_trials=20, long_episodes=12,
                  dt=0.04, reversal_dt=0.04, multi_dt=0.04, long_dt=0.04),
    "reduced": dict(seeds=3, reversal_seeds=12, vector_seeds=4, long_seeds=3,
                    interval_trials=500, reversal_trials=4000, multi_trials=1500,
                    vector_trials=3000, vector_reps=400, wm_trials=600, td_episodes=350,
                    beta_episodes=1200, beta_trials=1500, long_episodes=1500,
                    dt=0.02, reversal_dt=0.01, multi_dt=0.01, long_dt=0.01),
    "publication": dict(seeds=4, reversal_seeds=20, vector_seeds=8, long_seeds=4,
                        interval_trials=750, reversal_trials=6000, multi_trials=3000,
                        vector_trials=3000, vector_reps=800, wm_trials=900, td_episodes=500,
                        beta_episodes=1600, beta_trials=3000, long_episodes=2000,
                        dt=0.015, reversal_dt=0.005, multi_dt=0.005, long_dt=0.005),
}[RUN_PROFILE]

print({"profile": RUN_PROFILE, "device": "cpu", "workers": RESOLVED_WORKERS,
       "saving": SAVE_FIGURES, "archived_results": USE_ARCHIVED_RESULTS})

## Inline display, optional saving, and parallel execution

`finish` deliberately closes each figure after explicit display so Jupyter's inline
backend cannot render it a second time. Full archives are never selected merely
because a file happens to exist.

In [ ]:
def _safe_save(fig, entry):
    if not SAVE_FIGURES:
        return None
    if entry["provenance_class"] in {"reference-only", "placeholder", "external-gated"}:
        raise ValueError(f"refusing to publish {entry['provenance_class']} output: {entry['filename']}")
    out = OUTPUT_DIR.expanduser().resolve()
    if "manuscript" in str(out).lower() and not OVERWRITE:
        raise FileExistsError("writing into a manuscript directory requires OVERWRITE=True")
    out.mkdir(parents=True, exist_ok=True)
    path = out / entry["filename"]
    if path.exists() and not OVERWRITE:
        raise FileExistsError(path)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    return str(path)


def finish(fig, filename, *, seeds=None, source="live reduced computation"):
    if filename not in OWNED:
        raise KeyError(f"unregistered notebook figure: {filename}")
    entry = {"filename": filename, "provenance_class": (
        "published-aggregate" if source.startswith("archived") else "live-reduced")}
    saved = _safe_save(fig, entry)
    FIGURE_REPORT.append(dict(entry, runtime_profile=RUN_PROFILE, runtime_device="cpu",
                              seeds=seeds, saved_path=saved, display_status=source,
                              provenance_class=entry["provenance_class"],
                              method_provenance={"status": "proposed",
                                "established_basis": ["eligibility traces"],
                                "repository_adaptation": "repository-owned extension analysis",
                                "claim_limit": "exploratory simulation unless stated otherwise"}))
    display(fig)
    plt.close(fig)


def execute_specs(specs):
    # Run keyed module-level callables; never create a nested process pool.
    if not specs:
        return {}
    if RESOLVED_WORKERS == 1:
        return {key: fn(*args, **kwargs) for key, fn, args, kwargs in specs}
    ctx = get_context("spawn")
    with ProcessPoolExecutor(max_workers=min(RESOLVED_WORKERS, len(specs)), mp_context=ctx) as pool:
        futures = {key: pool.submit(fn, *args, **kwargs) for key, fn, args, kwargs in specs}
        return {key: future.result() for key, future in futures.items()}


def archived(name):
    if not USE_ARCHIVED_RESULTS:
        return None
    path = ROOT / "data" / "results" / name
    if not path.is_file():
        raise FileNotFoundError(f"requested archived result is absent: {path}")
    return np.load(path, allow_pickle=True).item()


def _ax_clean(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(True, color="0.93", lw=0.5)


GREEN, INDIGO, RED, GREY, GOLD = "#3aa07a", "#2f4b8f", "#c0392b", "#9aa6b2", "#e0a93b"

## Live reduced computations

The calls below use the same model functions as the full runners. Coarse conditions
are independent, seed-deterministic, and scheduled together. `smoke` changes only
the sample budget; it does not replace model output with a fixture.

In [ ]:
from mrl_trace.bandit import run_reversal, _summarize_reversal
from mrl_trace.extensions import (
    run_multitimescale, load_measured_tau, _mt_final, _mt_group_final,
    wm_isolated, run_wm_stc, _wm_final, run_device_td, _td_final,
    _dmax_one, _exp19_at_beta,
)
from mrl_trace.maze import _lh_run, _lh_summarize, LONG_HORIZON_CONDS
from mrl_trace.selectivity import run_interval_selectivity, aliasing_pair, run_interval, _final
from mrl_trace.stats import bootstrap_ci

S = SETTINGS
LIVE_SOURCE = f"live {RUN_PROFILE} computation"
BETAS = [1.0, 0.85, 0.54]

interval_result = archived("exp10_interval.npy")
reversal_result = archived("exp18_reversal.npy")
vector_result = archived("exp20_vector_timer.npy")
multi_result = archived("exp19_multitimescale.npy")

tau_pool, tau_source = load_measured_tau()
tau_grid = np.round(np.geomspace(max(0.5, tau_pool.min()), max(tau_pool.max(), 14.0),
                                 2 if RUN_PROFILE == "smoke" else 6), 2)

initial = []
if interval_result is None:
    initial.append(("interval", run_interval_selectivity, (),
                    dict(seeds=S["seeds"], trials=S["interval_trials"])))
if reversal_result is None:
    for condition in ("device", "abstract", "no_trace"):
        initial.append((f"reversal:{condition}", run_reversal, (condition,),
                        dict(B=S["reversal_seeds"], trials=S["reversal_trials"], dt=S["reversal_dt"])))
if vector_result is None:
    initial.append(("aliasing_pair", aliasing_pair, (),
                    dict(k=3, tau_leak=10.0, dt=S["dt"], reps=S["vector_reps"])))
if multi_result is None:
    rank_trials = min(S["multi_trials"], 1200 if RUN_PROFILE != "smoke" else 20)
    for tau in tau_grid:
        initial.append((f"rank:{tau:g}", run_multitimescale, ("best_single",),
                        dict(B=max(1, min(2, S["seeds"])), trials=rank_trials,
                             tau_arg=float(tau), tau_pool=tau_pool, dt=S["multi_dt"])))

initial_out = execute_specs(initial)
if interval_result is None:
    interval_result = initial_out["interval"]
if reversal_result is None:
    raw = {condition: initial_out[f"reversal:{condition}"]
           for condition in ("device", "abstract", "no_trace")}
    reversal_result = _summarize_reversal(raw, ("device", "abstract", "no_trace"),
                                          S["reversal_trials"], 0.75, 0.5)
    # Preserve the authored 500-trial bins at full scale; scale the bin only for the
    # reduced x-axis so the recovery panel remains resolved rather than collapsing to one point.
    flip = reversal_result["flips"][0]
    bin_width = 500 if S["reversal_trials"] >= 7000 else max(1, (S["reversal_trials"] - flip) // 7)
    for condition in ("device", "abstract", "no_trace"):
        segment = raw[condition][0][:, flip:].mean(0)
        reversal_result["recovery"][condition] = [
            float(segment[index:index + bin_width].mean())
            for index in range(0, len(segment), bin_width)
        ]
    reversal_result["recovery_bin"] = bin_width
if vector_result is None:
    tA, tB, tstar = initial_out["aliasing_pair"]
    vector_specs = []
    for readout in ("vector", "scalar", "no_trace"):
        vector_specs.append((f"main:{readout}", run_interval, (readout,),
                             dict(B=S["vector_seeds"], trials=S["vector_trials"],
                                  tA=tA, tB=tB, dt=S["dt"])))
    seps, ks = [12.0, 6.0, 3.0, 1.5], [3, 5, 8]
    sweep_trials = max(10, S["vector_trials"] // 3)
    for sep in seps:
        for depth in ks:
            for readout in ("scalar", "vector"):
                vector_specs.append((f"sweep:{sep}:{depth}:{readout}", run_interval, (readout,),
                                     dict(B=S["vector_seeds"], k=depth, trials=sweep_trials,
                                          tA=9.0-sep/2, tB=9.0+sep/2, dt=S["dt"])))
    vector_out = execute_specs(vector_specs)
    finals = {key: _final(vector_out[f"main:{key}"]) for key in ("vector", "scalar", "no_trace")}
    vector_result = {
        "finals": finals,
        "ci": {key: bootstrap_ci(value) for key, value in finals.items()},
        "sweep": {f"{sep}_{depth}": {
            readout: float(_final(vector_out[f"sweep:{sep}:{depth}:{readout}"]).mean())
            for readout in ("scalar", "vector")}
            for sep in seps for depth in ks},
        "seps": seps, "ks": ks, "tau_leak": 10.0,
        "tA": tA, "tB": tB, "tstar": tstar,
        "seeds": S["vector_seeds"], "trials": S["vector_trials"],
        "dt": S["dt"], "alias_reps": S["vector_reps"], "chance": 0.5, "crit": 0.75,
    }

if multi_result is None:
    best_tau = max(tau_grid, key=lambda tau: float(_mt_final(initial_out[f"rank:{tau:g}"][0]).mean()))
    conds = [
        ("hetero_raw", "hetero_measured", "none"),
        ("hetero_homeo", "hetero_measured", "homeo"),
        ("hetero_oracle", "hetero_measured", "oracle"),
        ("best_single", "best_single", "none"),
        ("no_trace", "no_trace", "none"),
    ]
    specs = []
    for label, kind, norm in conds:
        specs.append((label, run_multitimescale, (kind,),
                      dict(B=S["seeds"], trials=S["multi_trials"],
                           tau_arg=(float(best_tau) if kind == "best_single" else None),
                           tau_pool=tau_pool, elig_norm=norm, dt=S["multi_dt"])))
    out = execute_specs(specs)
    finals, cis, groups = {}, {}, {}
    for label, _, _ in conds:
        rewards, grouped, *_ = out[label]
        finals[label] = _mt_final(rewards)
        cis[label] = bootstrap_ci(finals[label])
        groups[label] = (float(np.nanmean(_mt_group_final(grouped["short"]))),
                         float(np.nanmean(_mt_group_final(grouped["long"]))))
    multi_result = dict(finals=finals, ci=cis, grp=groups, tau_source=tau_source,
                        tau_pool=tau_pool, best_tau=float(best_tau), chance=0.5, crit=0.75,
                        seeds=S["seeds"], trials=S["multi_trials"], dt=S["multi_dt"])
else:
    best_tau = float(multi_result.get("best_tau", np.median(tau_pool)))

print("interval/reversal/vector/multi complete", {"best_tau": best_tau, "tau_source": tau_source})

In [ ]:
# Working-memory/tagging and device-TD cells are independent across retention or track length.
wm_result = archived("exp22_wm_stc.npy")
td_result = archived("exp23_device_td.npy")

specs = []
if wm_result is None:
    tau_band = [0.8, 1.3, 2.0, 3.6, 6.0, 10.0]
    for tau in tau_band:
        specs.extend([
            (f"wm:{tau}", wm_isolated, (tau,),
             dict(B=S["seeds"], trials=max(10, S["wm_trials"] // 4), dt=S["dt"])),
            (f"stc:{tau}", run_wm_stc, (10.0, tau),
             dict(B=S["seeds"], trials=S["wm_trials"], dt=S["dt"])),
            (f"shared:{tau}", run_wm_stc, (tau, tau),
             dict(B=S["seeds"], trials=S["wm_trials"], dt=S["dt"], shared_device=True)),
        ])
if td_result is None:
    L_grid = [5, 10, 15, 20]
    schemes = ["no_trace", "reinforce", "td_actor_critic", "td_no_homeo"]
    for length in L_grid:
        for scheme in schemes:
            specs.append((f"td:{scheme}:{length}", run_device_td, (scheme,),
                          dict(L=length, B=S["seeds"], episodes=S["td_episodes"], dt=S["dt"])))

out = execute_specs(specs)
if wm_result is None:
    wm_result = dict(B=S["seeds"], trials=S["wm_trials"], tau_band=tau_band, chance=0.5,
                     wm={}, stc={}, one_shared={})
    for tau in tau_band:
        wm_result["wm"][tau] = out[f"wm:{tau}"]
        wm_result["stc"][tau] = _wm_final(out[f"stc:{tau}"])
        wm_result["one_shared"][tau] = _wm_final(out[f"shared:{tau}"])
if td_result is None:
    td_result = dict(B=S["seeds"], episodes=S["td_episodes"], L_grid=L_grid,
                     crit=0.75, final={}, curve={})
    for length in L_grid:
        for scheme in schemes:
            rewards, values = out[f"td:{scheme}:{length}"]
            td_result["final"][(scheme, length)] = _td_final(rewards)
            td_result["curve"][(scheme, length)] = rewards.mean(0)

print("WM/STC and device-TD complete")

In [ ]:
# Dispersion and long-horizon grids share the same spawn-safe outer pool.
beta_result = archived("exp21_beta_sensitivity.npy")
long_result = archived("exp15_long_horizon.npy")
specs = []

if beta_result is None:
    dmax_taus = [1.0, 2.0, 5.0, 10.0, 20.0]
    dmax_delays = [2, 5, 10, 20, 40, 80, 160]
    for beta in BETAS:
        for tau in dmax_taus:
            job = (tau, beta, S["seeds"], S["beta_episodes"], dmax_delays, S["dt"])
            specs.append((f"dmax:{beta}:{tau}", _dmax_one, (job,), {}))
    for beta in BETAS[1:]:  # beta=1 reuses the already-computed multi-timescale cells
        job = (beta, S["seeds"], S["beta_trials"], tau_pool, best_tau, S["multi_dt"])
        specs.append((f"exp19:{beta}", _exp19_at_beta, (job,), {}))

if long_result is None:
    long_L, long_A = [3, 5, 8, 12], [2, 3]
    long_jobs = [(length, actions, condition, seed, S["long_episodes"], 2.0, 10.0, S["long_dt"])
                 for length in long_L for actions in long_A
                 for condition in LONG_HORIZON_CONDS for seed in range(S["long_seeds"])]
    for index, job in enumerate(long_jobs):
        specs.append((f"long:{index}", _lh_run, (job,), {}))

out = execute_specs(specs)

if beta_result is None:
    dmax = {}
    for beta in BETAS:
        values = np.asarray([out[f"dmax:{beta}:{tau}"][1] for tau in dmax_taus], float)
        taus = np.asarray(dmax_taus, float)
        slope = float(np.sum(taus * values) / np.sum(taus * taus))
        pred = slope * taus
        r2 = 1 - np.sum((values - pred) ** 2) / max(np.sum((values - values.mean()) ** 2), 1e-9)
        dmax[beta] = dict(taus=dmax_taus, dmax=values.tolist(), k=slope,
                          r2=float(r2), monotone=bool(np.all(np.diff(values) >= 0)))
    exp19_beta = {
        "1.0": {key: (float(np.mean(multi_result["finals"][key])), multi_result["ci"][key])
                for key in ("hetero_raw", "hetero_homeo", "best_single")}
    }
    for beta in BETAS[1:]:
        exp19_beta[str(beta)] = out[f"exp19:{beta}"][1]
    beta_result = dict(dmax=dmax, exp19=exp19_beta, betas=BETAS,
                       seeds=S["seeds"], tau_source=tau_source)

if long_result is None:
    raw = [out[f"long:{index}"] for index in range(len(long_jobs))]
    long_result = _lh_summarize(raw, long_L, long_A, S["long_episodes"],
                                2.0, 10.0, S["long_seeds"])
    long_result["dt"] = S["long_dt"]

print("dispersion and long-horizon grids complete")

## Interval selectivity

Explores how the proposed surrogate trace can generate interval-dependent credit in simulation.

The first panel is generated analytically from the fitted trace. The second is the
learned selectivity of the live reduced runs; it is not reconstructed from summary
statistics.

In [ ]:
from mrl_trace.selectivity import trace_kernel, abstract_kernel

d = interval_result
V, taus, tstar, D_grid, Scurve = d["V"], d["taus"], d["tstar"], d["D_grid"], d["Scurve"]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(7.2, 3.0))
t = np.arange(0, 60, 0.05)
kd = trace_kernel(t, 10.0, V=V); kd = kd / kd.max()
ke = abstract_kernel(t, 10.0)
axA.plot(t, kd, color=GREEN, lw=1.9, label="device (peaked)")
axA.plot(t, ke, color=INDIGO, lw=1.7, ls=(0, (4, 1.5)), label="exponential (monotone)")
axA.axvline(tstar[10.0], ls=":", color=GOLD, lw=1.0)
axA.annotate(r"$t^\ast$", (tstar[10.0], 1.02), color=GOLD, fontsize=9, ha="center")
axA.axvline(0.3, ls=":", color="#888", lw=1.0)
axA.set_xlabel("lag before reward (s)"); axA.set_ylabel(r"eligibility $e$")
axA.set_xlim(0, 60); axA.set_ylim(0, 1.08)
axA.set_title(r"(a) trace shape ($\tau_{\rm leak}=10$ s)", loc="left", fontsize=9.5, fontweight="bold")
axA.grid(True, ls=":", lw=0.5, color="#c8d0d8"); axA.set_axisbelow(True)
axA.legend(frameon=False, fontsize=8); axA.spines[["top", "right"]].set_visible(False)
cols = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]
for tau, colour in zip(taus, cols):
    axB.plot(D_grid, Scurve[tau], marker="o", ms=4, lw=1.6, color=colour,
             label=fr"$\tau_{{\rm leak}}={tau:g}$ s")
axB.axhline(1.0, ls="--", color="#888", lw=1.0)
axB.annotate("selectivity = 1\n(no preference)", (D_grid[-1], 1.0), fontsize=7,
             color="#666", ha="right", va="bottom")
axB.set_xscale("log"); axB.set_xlabel(r"design interval $D$ (s)")
axB.set_ylabel(r"learned $w_{\rm pref}/w_{\rm late}$")
axB.set_xticks(D_grid); axB.set_xticklabels([f"{value:g}" for value in D_grid])
axB.set_title("(b) learned interval selectivity", loc="left", fontsize=9.5, fontweight="bold")
axB.grid(True, which="major", ls=":", lw=0.5, color="#c8d0d8"); axB.set_axisbelow(True)
axB.legend(frameon=False, fontsize=8); axB.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
finish(fig, "fig_interval.png", seeds=d.get("seeds"),
       source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Reversal learning

Explores acquisition and reversal across the same retention-matched simulation controls.
The result is descriptive: all conditions and recovery times are reported, including null effects.


In [ ]:
d = reversal_result
cur, rec, flips = d["curves"], d["recovery"], d["flips"]
col = {"device": GREEN, "abstract": INDIGO, "no_trace": GREY}
lab = {"device": "device trace", "abstract": "abstract (matched $\\tau$)", "no_trace": "no trace"}
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for key in ("device", "abstract", "no_trace"):
    y = np.asarray(cur[key]); ax[0].plot(np.arange(len(y)), y, color=col[key], lw=1.8, label=lab[key])
for flip in flips:
    ax[0].axvline(flip, color="0.4", ls="--", lw=1.2)
ax[0].text(flips[0], 1.02, "reversal", fontsize=8, ha="center", color="0.3")
ax[0].axhline(0.5, color="0.6", ls=":", lw=1.0)
ax[0].set_xlabel("trial"); ax[0].set_ylabel("reward rate"); ax[0].set_ylim(0, 1.08)
ax[0].set_title("(a) Acquire, reverse, re-acquire", fontsize=10, loc="left")
handles, labels = ax[0].get_legend_handles_labels(); _ax_clean(ax[0])
bin_width = d.get("recovery_bin", 500)
for key in ("device", "abstract", "no_trace"):
    y = np.asarray(rec[key]); ax[1].plot(np.arange(len(y)) * bin_width, y, "o-", color=col[key], lw=1.8, ms=4)
ax[1].axhline(0.5, color="0.6", ls=":", lw=1.0)
ax[1].set_xlabel("trials after reversal"); ax[1].set_ylabel("reward rate"); ax[1].set_ylim(0, 1.08)
ratio = d["reacq_cost"]["ratio"]
title = "(b) Post-reversal recovery"
ax[1].set_title(title, fontsize=10, loc="left"); _ax_clean(ax[1])
fig.suptitle("Exploratory reversal sensitivity across trace conditions", fontsize=10.5)
fig.legend(handles, labels, ncol=3, fontsize=8, frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.06, 1, 1))
finish(fig, "fig_reversal.png", seeds=d.get("seeds"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Multi-timescale credit

Explores a simulated mixture parameterized by the direct held-bias retention distribution.
It does not establish a general advantage for heterogeneous traces.


In [ ]:
d = multi_result
order = ["hetero_raw", "hetero_homeo", "hetero_oracle", "best_single", "no_trace"]
lab = {"hetero_raw": "measured spread\n(raw)", "hetero_homeo": "spread +\nhomeostasis",
       "hetero_oracle": "spread +\n$\\tau$-oracle", "best_single": "best single\n$\\tau$", "no_trace": "no trace"}
cols = {"hetero_raw": GREY, "hetero_homeo": GREEN, "hetero_oracle": INDIGO,
        "best_single": GOLD, "no_trace": "0.7"}
means = [float(np.mean(d["finals"][key])) for key in order]
cis = [d["ci"][key] for key in order]
lo = [means[i] - cis[i][0] for i in range(len(order))]
hi = [cis[i][1] - means[i] for i in range(len(order))]
fig, ax = plt.subplots(figsize=(5.8, 3.8)); x = np.arange(len(order))
ax.bar(x, means, color=[cols[key] for key in order], yerr=np.vstack([lo, hi]),
       capsize=3, error_kw=dict(lw=1, ecolor="0.3"))
ax.axhline(d["chance"], color=RED, ls="--", lw=1.1, label=f"chance ({d['chance']:.2f})")
ax.axhline(d["crit"], color="0.5", ls=":", lw=1.0, label=f"criterion ({d['crit']:.2f})")
ax.set_xticks(x); ax.set_xticklabels([lab[key] for key in order], fontsize=7.5)
ax.set_ylabel("final reward rate"); ax.set_ylim(0, 1.12)
ax.set_title("Exploratory multi-timescale credit across\nthe held-bias $\\tau$ distribution", fontsize=9.5)
ax.legend(fontsize=7, frameon=False, loc="upper center", ncol=2,
          columnspacing=1.4, handlelength=1.6)
_ax_clean(ax); fig.tight_layout()
finish(fig, "fig_multitimescale.png", seeds=d.get("seeds"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Cascade-vector timer

Explores a vector readout of simulated cascade stages. The present two-terminal
device does not expose those stages, so this is future work rather than device evidence.


In [ ]:
d = vector_result; fin, cis = d["finals"], d["ci"]
tA, tB = d.get("tA", 10.5), d.get("tB", 30.5)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
order = ["vector", "scalar", "no_trace"]
cols = {"vector": GREEN, "scalar": INDIGO, "no_trace": GREY}
lab = {"vector": "full vector", "scalar": "scalar (last stage)", "no_trace": "no trace"}
means = [float(np.mean(fin[key])) for key in order]
lo = [means[i] - cis[order[i]][0] for i in range(3)]
hi = [cis[order[i]][1] - means[i] for i in range(3)]
ax[0].bar(np.arange(3), means, color=[cols[key] for key in order], yerr=np.vstack([lo, hi]),
          capsize=3, error_kw=dict(lw=1, ecolor="0.3"))
ax[0].axhline(0.5, color=RED, ls="--", lw=1.1, label="chance (0.50)")
ax[0].axhline(0.75, color="0.6", ls=":", lw=1.0, label="criterion (0.75)")
ax[0].set_xticks(range(3)); ax[0].set_xticklabels([lab[key] for key in order], fontsize=8)
ax[0].set_ylabel("reward rate"); ax[0].set_ylim(0, 1.12)
ax[0].set_title(f"(a) Aliasing pair $t_A$={tA:.0f}s, $t_B$={tB:.0f}s", fontsize=10, loc="left")
ax[0].legend(fontsize=7, frameon=False, loc="upper right", handlelength=1.6); _ax_clean(ax[0])
for k in d["ks"]:
    advantage = [d["sweep"][f"{sep}_{k}"]["vector"] - d["sweep"][f"{sep}_{k}"]["scalar"] for sep in d["seps"]]
    ax[1].plot(d["seps"], advantage, "o-", color=(GREEN if k == d["ks"][0] else INDIGO),
               lw=1.8, ms=5, label=f"$k$={k}")
ax[1].axhline(0, color="0.5", ls=":", lw=1.0)
ax[1].set_xlabel("interval separation (s)"); ax[1].set_ylabel("vector $-$ scalar advantage")
ax[1].set_title("(b) Advantage is noise-bounded, not $k$-bounded", fontsize=10, loc="left")
ax[1].legend(fontsize=8, frameon=False); _ax_clean(ax[1])
fig.suptitle("Exploratory cascade-vector timer (simulation; requires stage-level readout)", fontsize=9.5)
fig.tight_layout()
finish(fig, "fig_vector_timer.png", seeds=d.get("seeds"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Exploratory assumed capture model

The volatile tag is simulated, but the capture/commit transition is explicitly assumed.
The measured conductance ladder only quantizes that assumed update; it does not
demonstrate synaptic tagging and capture in the two-terminal device.


In [ ]:
from mrl_trace.capture import TwoStateSynapse, capture_curve, load_measured_conductance_ladder

INK, TAG, CAP, NEU = "#2b2b2b", GREEN, RED, GREY
PAL = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]
ladder = load_measured_conductance_ladder()

def panel_a(ax):
    V, tau_leak, dt = 0.9, 2.0, 0.02
    t = np.arange(0.0, 12.0, dt); coincidence_at = 1.0
    syn_in = TwoStateSynapse(V=V, tau_leak=tau_leak, dt=dt, ladder=ladder)
    out_in = syn_in.run(t, coincidence_at=coincidence_at, reward_at=2.5, reward=1.0)
    syn_late = TwoStateSynapse(V=V, tau_leak=tau_leak, dt=dt, ladder=ladder)
    out_late = syn_late.run(t, coincidence_at=coincidence_at, reward_at=10.0, reward=1.0)
    trace = out_in["v"] / out_in["v"].max()
    ax.fill_between(t, 0, trace, color=TAG, alpha=0.18, zorder=1)
    ax.plot(t, trace, color=TAG, lw=2.0, zorder=3, label=r"volatile tag $v(t)$ (fitted)")
    ax.axvspan(1.0, 1.2, color=GOLD, alpha=0.30, zorder=0)
    ax.text(1.1, 1.06, "coincidence", color=GOLD, fontsize=8.5, ha="center", va="bottom")
    for delay, label, ok in ((1.5, "reward within window", True), (9.0, "reward too late", False)):
        x = 1.0 + delay; ax.axvline(x, color=CAP, lw=1.6, ls=(0, (4, 2)), alpha=1.0 if ok else 0.5)
        ax.text(x, 1.18 if ok else 1.06, label, color=CAP, fontsize=8.2, ha="center", va="bottom", alpha=1.0 if ok else 0.7)
    axr = ax.twinx()
    g_in = (out_in["G"] - out_in["G0"]) / syn_in.g_step
    g_late = (out_late["G"] - out_late["G0"]) / syn_late.g_step
    axr.plot(t, g_in, color=CAP, lw=2.2, label="assumed committed weight (in window)")
    axr.plot(t, g_late, color=NEU, lw=2.0, ls=(0, (2, 1.5)), label="assumed committed weight (too late)")
    axr.set_ylabel(r"assumed $\Delta G$ (ladder-referenced steps)", color=INK, fontsize=9)
    axr.set_ylim(-0.05, max(0.6, g_in.max() * 1.25)); axr.tick_params(labelsize=8)
    ax.set_xlabel("time (s)"); ax.set_ylabel(r"normalised tag $v(t)$", color=TAG)
    ax.set_ylim(0, 1.32); ax.set_xlim(0, 12); ax.tick_params(labelsize=8)
    ax.set_title("(a) assumed commit is gated by the simulated tag", fontsize=10.5, fontweight="bold", loc="left")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axr.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7.6, loc="upper right", framealpha=0.92)

def panel_b(ax):
    delays = np.linspace(0.0, 18.0, 46)
    taus = [1.3, 4.0, 8.0]
    labels = [r"$\tau_{\mathrm{leak}}=1.3$ s (ITO)", r"$\tau_{\mathrm{leak}}=4$ s", r"$\tau_{\mathrm{leak}}=8$ s"]
    for tau, label, colour in zip(taus, labels, PAL):
        result = capture_curve(delays=delays, tau_leak=tau, V=0.9, reward=1.0, dt=0.02, ladder=ladder)
        norm = result["dG"] / result["dG"].max()
        ax.plot(result["D"], norm, color=colour, lw=2.2, label=label)
        ax.axvline(tau, color=colour, lw=1.0, ls=(0, (1, 2)), alpha=0.6)
    ax.set_xlabel(r"action$\rightarrow$reward delay $D$ (s)"); ax.set_ylabel(r"committed change $\Delta G$ (normalised)")
    ax.set_xlim(0, 18); ax.set_ylim(bottom=0); ax.tick_params(labelsize=8)
    ax.set_title("(b) modeled commit window tracks tag lifetime", fontsize=10.5, fontweight="bold", loc="left")
    ax.legend(fontsize=8.0, loc="upper right", framealpha=0.92)
    ax.text(0.5, -0.22, "Within this simulation, a longer tag extends the modeled commit window",
            transform=ax.transAxes, fontsize=8.0, ha="center", color=NEU, style="italic")

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.2, 4.3)); panel_a(axA); panel_b(axB)
fig.subplots_adjust(left=0.06, right=0.93, top=0.92, bottom=0.12, wspace=0.42)
finish(fig, "fig_capture.png", source="exploratory assumed capture model with ladder-referenced conductance steps")

## Working memory and tagging/capture

Explores a functional analogy between simulated short-term memory and an assumed
tag/commit mechanism. This is not evidence of classical biological synaptic tagging
and capture, which occurs on different timescales and requires additional biology.


In [ ]:
d = wm_result; taus = np.asarray(d["tau_band"], float)
def _mci(mapping):
    means = np.array([np.asarray(mapping[tau]).mean() for tau in taus])
    lows = np.array([np.percentile(np.asarray(mapping[tau]), 2.5) for tau in taus])
    highs = np.array([np.percentile(np.asarray(mapping[tau]), 97.5) for tau in taus])
    return means, lows, highs
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10, 3.8))
for key, colour, label in [("wm", INDIGO, "cue hold (WM)"), ("stc", GREEN, "credit tag")]:
    mean, low, high = _mci(d[key]); axA.plot(taus, mean, "-o", color=colour, lw=1.8, ms=4, label=label)
    axA.fill_between(taus, low, high, color=colour, alpha=0.15)
axA.axhline(0.75, ls="--", color=GREY, lw=1.0); axA.set_xscale("log")
axA.set_xticks(taus); axA.set_xticklabels([f"{tau:g}" for tau in taus])
axA.set_xlabel(r"retention $\tau_{leak}$ (s)"); axA.set_ylabel("task performance")
axA.set_ylim(0.45, 1.02); axA.set_title("(a) per-role requirement", fontsize=10)
axA.legend(fontsize=8, loc="lower right", frameon=False); _ax_clean(axA)
mean, low, high = _mci(d["one_shared"])
axB.axvspan(1.3, 6.0, color=GREEN, alpha=0.07)
axB.plot(taus, mean, "-o", color=GREEN, lw=1.8, ms=4, label="one device, both roles")
axB.fill_between(taus, low, high, color=GREEN, alpha=0.15)
axB.axhline(0.75, ls="--", color=GREY, lw=1.0); axB.axhline(0.5, ls=":", color=GREY, lw=1.0)
axB.set_xscale("log"); axB.set_xticks(taus); axB.set_xticklabels([f"{tau:g}" for tau in taus])
axB.set_xlabel(r"retention $\tau_{leak}$ (s)"); axB.set_ylabel("joint-task performance")
axB.set_ylim(0.45, 1.02); axB.set_title("(b) single substrate, both roles", fontsize=10)
axB.legend(fontsize=8, loc="lower right", frameon=False); _ax_clean(axB)
fig.tight_layout()
finish(fig, "fig_wm_stc.png", seeds=d.get("B"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Exploratory device-parameterized temporal-difference learning

Evaluates a repository-specific actor--critic simulation parameterized by the trace
surrogate. The device does not itself compute the complete TD update, and the method
is not claimed to outperform the simpler comparator.


In [ ]:
d = td_result; lengths = np.asarray(d["L_grid"], float)
series = [("reinforce", INDIGO, "policy gradient (REINFORCE)"),
          ("td_actor_critic", GREEN, "proposed trace-based TD actor-critic"),
          ("td_no_homeo", GOLD, "TD, no eligibility homeostasis"),
          ("no_trace", GREY, "no-trace (chance)")]
def _td_mci(scheme):
    mean = np.array([np.asarray(d["final"][(scheme, length)]).mean() for length in d["L_grid"]])
    low = np.array([np.percentile(np.asarray(d["final"][(scheme, length)]), 2.5) for length in d["L_grid"]])
    high = np.array([np.percentile(np.asarray(d["final"][(scheme, length)]), 97.5) for length in d["L_grid"]])
    return mean, low, high
fig, ax = plt.subplots(figsize=(6.6, 4.0))
for scheme, colour, label in series:
    mean, low, high = _td_mci(scheme)
    ax.plot(lengths, mean, "-o", color=colour, lw=1.7, ms=4, label=label)
    ax.fill_between(lengths, low, high, color=colour, alpha=0.13)
ax.axhline(d.get("crit", 0.75), ls="--", color=GREY, lw=1.0)
ax.set_xlabel("corridor length $L$"); ax.set_ylabel("final goal-reach rate")
ax.set_xticks(lengths); ax.set_ylim(0, 1.03)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False)
ax.set_title("Exploratory trace-based TD and REINFORCE comparison"); _ax_clean(ax); fig.tight_layout()
finish(fig, "fig_device_td.png", seeds=d.get("B"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Dispersion sensitivity

Tests simulation sensitivity to selected compressed-exponential shape parameters.
This is not independent physical validation of a retention--delay law.


In [ ]:
dmax, exp19_beta = beta_result["dmax"], beta_result["exp19"]
def beta_key(mapping, beta):
    return beta if beta in mapping else str(beta)

colours = [plt.cm.viridis(value) for value in (0.15, 0.5, 0.85)]
fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.3))
for beta, colour in zip(BETAS, colours):
    result = dmax[beta_key(dmax, beta)]
    ax[0].plot(result["taus"], result["dmax"], "o-", color=colour, lw=1.9, ms=5,
               label=rf"$\beta_{{\mathrm{{leak}}}}={beta:g}$ ($k={result['k']:.1f}$)")
ax[0].set_xlabel(r"retention $\tau_{\mathrm{leak}}$ (s)"); ax[0].set_ylabel(r"$D_{\max}$ (s)")
ax[0].set_title(r"(a) simulated threshold-delay sensitivity", fontsize=9.5, loc="left")
ax[0].legend(fontsize=7.5, frameon=False, loc="lower right"); _ax_clean(ax[0])
x = np.arange(len(BETAS)); width = 0.38
homeo = [exp19_beta[beta_key(exp19_beta, beta)]["hetero_homeo"][0] for beta in BETAS]
best = [exp19_beta[beta_key(exp19_beta, beta)]["best_single"][0] for beta in BETAS]
ax[1].bar(x-width/2, homeo, width, color=INDIGO, label="hetero + homeo")
ax[1].bar(x+width/2, best, width, color=GREY, label=r"best single-$\tau$")
ax[1].axhline(0.5, color=RED, ls="--", lw=1.1, label="chance (0.50)")
ax[1].set_xticks(x); ax[1].set_xticklabels([rf"$\beta={beta:g}$" for beta in BETAS])
ax[1].set_ylabel("final reward rate"); ax[1].set_ylim(0, 1.42)
ax[1].set_title("(b) multi-timescale coupling", fontsize=9.5, loc="left")
ax[1].legend(fontsize=7.0, frameon=False, loc="upper center", ncol=3, columnspacing=1.0,
             handlelength=1.3, handletextpad=0.4, bbox_to_anchor=(0.5, 1.02)); _ax_clean(ax[1])
fig.tight_layout()
finish(fig, "fig_betaval.png", seeds=beta_result.get("seeds"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

In [ ]:
slopes = [dmax[beta_key(dmax, beta)]["k"] for beta in BETAS]
raw = [exp19_beta[beta_key(exp19_beta, beta)]["hetero_raw"][0] for beta in BETAS]
homeo = [exp19_beta[beta_key(exp19_beta, beta)]["hetero_homeo"][0] for beta in BETAS]
labels = [r"$\beta{=}1$"+"\n(single-rate,\nmain text)",
          r"$\beta{=}0.85$"+"\n(preliminary,\nnear-zero read)",
          r"$\beta{=}0.54$"+"\n(held-bias,\nstress)"]
x = np.arange(len(BETAS)); fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.3))
ax[0].bar(x, slopes, color=colours)
for xpos, slope in zip(x, slopes):
    ax[0].text(xpos, slope + 0.15, f"{slope:.1f}", ha="center", fontsize=8)
ax[0].set_xticks(x); ax[0].set_xticklabels(labels, fontsize=7)
ax[0].set_ylabel(r"$D_{\max}/\tau$ slope $k$"); ax[0].set_ylim(0, max(slopes) + 1.5)
ax[0].set_title("(a) Simulated retention-delay sensitivity", fontsize=9.5, loc="left"); _ax_clean(ax[0])
width = 0.38
ax[1].bar(x-width/2, raw, width, color=GREY, label="measured spread (raw)")
ax[1].bar(x+width/2, homeo, width, color=INDIGO, label="spread + homeostasis")
ax[1].axhline(0.5, color=RED, ls="--", lw=1.1, label="chance (0.50)")
ax[1].set_xticks(x); ax[1].set_xticklabels(labels, fontsize=7)
ax[1].set_ylabel("final reward rate"); ax[1].set_ylim(0, 1.12)
ax[1].set_title("(b) Multi-timescale coupling", fontsize=9.5, loc="left")
ax[1].legend(fontsize=7.5, frameon=False, loc="upper center"); _ax_clean(ax[1])
fig.suptitle(r"Sensitivity across selected dispersion values", fontsize=10)
fig.tight_layout()
finish(fig, "fig_beta_sensitivity.png", seeds=beta_result.get("seeds"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Longer-horizon credit

Explores long-horizon sensitivity in the repository task. Comparator labels identify
adaptations precisely; this panel does not establish algorithmic superiority.


In [ ]:
d = long_result; summary, lengths, conditions = d["summary"], d["L"], d["conds"]
colours = {"device": GREEN, "eprop": INDIGO, "abstract": "0.5", "no_trace": GREY}
labels = {"device": "device surrogate", "eprop": "shallow e-prop-style", "abstract": "exponential control", "no_trace": "no trace"}
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for panel, actions in enumerate(d["A"]):
    for condition in conditions:
        mean = [np.ravel(summary[(length, actions, condition)])[0] for length in lengths]
        low = [np.ravel(summary[(length, actions, condition)])[1] for length in lengths]
        high = [np.ravel(summary[(length, actions, condition)])[2] for length in lengths]
        ax[panel].fill_between(lengths, low, high, color=colours[condition], alpha=0.18, lw=0)
        ax[panel].plot(lengths, mean, "o-", color=colours[condition], lw=1.8, ms=4, label=labels[condition])
    ax[panel].axhline(1.0/actions, color="0.6", ls=":", lw=1.0)
    ax[panel].set_xlabel("trajectory length $L$"); ax[panel].set_ylabel("reward rate")
    ax[panel].set_ylim(0, 1.08); ax[panel].set_title(f"({'ab'[panel]}) $A$={actions} actions", fontsize=10, loc="left")
    _ax_clean(ax[panel])
    if panel == 0:
        ax[panel].legend(fontsize=8, frameon=False, loc="center right")
fig.suptitle("Exploratory longer-horizon credit sensitivity (shaded 95\\% CI)", fontsize=9.5)
fig.tight_layout()
finish(fig, "fig_long_horizon.png", seeds=d.get("seeds"), source=("archived full sweep" if USE_ARCHIVED_RESULTS else LIVE_SOURCE))

## Notebook report

In [ ]:
assert len(FIGURE_REPORT) == 10
assert len({row["filename"] for row in FIGURE_REPORT}) == 10
FIGURE_REPORT